# 03: Trajectory Evaluation

**Thomson Reuters | Agentic AI Evaluation Bootcamp**

In Notebook 02 we scored *what* the agent answered. Here we score *how* it behaved.

## What You'll Do

1. Define 5 **trace-level evaluators** that inspect the agent's tool call sequence
2. Run a **two-pass experiment**: output quality (answer scores) + trajectory quality (behavior scores)
3. See both output-level and trace-level scores side by side
4. Identify patterns — does the agent search before fetching? Does it cite sources? Is it redundant?

## Why Trajectory Evaluation?

An agent can get the right answer in a terrible way (lucky guess, redundant searches, no sources cited).
It can also fail despite doing everything right (great search pattern, but the web page was wrong).
Trajectory evaluation separates **answer quality** from **behavioral quality** — both matter for a
production system.

## Our 5 Trace Evaluators

| Metric | What it checks | Score |
|--------|---------------|-------|
| `trace_tool_call_count` | How many tools did the agent use? | Integer count |
| `trace_search_before_fetch` | Did `google_search` always precede `web_fetch`? | 1.0 / 0.0 |
| `trace_has_sources` | Did the agent cite at least one URL? | 1.0 / 0.0 |
| `trace_redundant_search_ratio` | Fraction of duplicate search queries | 0.0 = no redundancy |
| `trace_latency_sec` | Total wall-clock time in seconds | Float |

**No hand-crafted ground truth needed.** Each metric is a deterministic rule on the actual trace.

## Prerequisites

- Notebook 02 completed — dataset already uploaded to Langfuse as `TR-DeepSearchQA`
- All keys in `~/.env`

## 1. Setup & Configuration

In [ ]:
import json
import os
import tempfile
from pathlib import Path
from typing import Any

import pandas as pd
from aieng.agent_evals.async_client_manager import AsyncClientManager
from aieng.agent_evals.evaluation import run_experiment_with_trace_evals
from aieng.agent_evals.evaluation.graders.config import LLMRequestConfig
from aieng.agent_evals.evaluation.trace import extract_trace_metrics
from aieng.agent_evals.evaluation.types import TraceWaitConfig
from aieng.agent_evals.knowledge_qa import DeepSearchQADataset, KnowledgeGroundedAgent
from aieng.agent_evals.knowledge_qa.deepsearchqa_grader import DeepSearchQAResult, evaluate_deepsearchqa_async
from aieng.agent_evals.langfuse import init_tracing, upload_dataset_to_langfuse
from dotenv import load_dotenv
from IPython.display import HTML, display
from langfuse.experiment import Evaluation
from rich.console import Console
from rich.panel import Panel
from rich.table import Table

# Set working directory to repo root
if Path('').absolute().name == 'eval-agents':
    print(f'Working directory: {Path("").absolute()}')
else:
    os.chdir(Path('').absolute().parent.parent)
    print(f'Working directory set to: {Path("").absolute()}')

load_dotenv(verbose=True)
init_tracing()
console = Console(width=100)

# ── Configuration ──────────────────────────────────────────────────────────────
CATEGORY        = 'Finance & Economics'       # Must match Notebook 02
NUM_SAMPLES     = 5                           # Must match Notebook 02
DATASET_NAME    = 'TR-DeepSearchQA'           # Must match Notebook 02
EXPERIMENT_NAME = 'tr-trajectory-v1'          # New name for this trajectory run
# ───────────────────────────────────────────────────────────────────────────────

console.print(Panel(
    f'[bold]Category:[/bold]   {CATEGORY}\n'
    f'[bold]Samples:[/bold]    {NUM_SAMPLES}\n'
    f'[bold]Dataset:[/bold]    {DATASET_NAME}\n'
    f'[bold]Experiment:[/bold] {EXPERIMENT_NAME}',
    title='TR Knowledge QA — Notebook 03',
    border_style='cyan'
))

## 2. Agent Task & Output-Level Evaluator

Same as Notebook 02 — we need these for the two-pass experiment.

In [ ]:
async def agent_task(*, item: Any, **kwargs: Any) -> str:
    """Run the PlanReAct Knowledge Agent on one dataset item."""
    agent = KnowledgeGroundedAgent(enable_planning=True)
    response = await agent.answer_async(item.input)
    client_manager = AsyncClientManager.get_instance()
    client_manager.langfuse_client.update_current_span(
        metadata=response.model_dump(exclude={'text'})
    )
    return response.text


async def deepsearchqa_evaluator(
    *,
    input: str,
    output: str,
    expected_output: str,
    metadata: dict[str, Any] | None = None,
    **kwargs: Any,
) -> list[Evaluation]:
    """LLM-as-judge: scores the agent's answer against ground truth."""
    answer_type = (metadata or {}).get('answer_type', 'Set Answer')
    try:
        result =  evaluate_deepsearchqa_async(
            question=input,
            answer=str(output),
            ground_truth=expected_output,
            answer_type=answer_type,
            model_config=LLMRequestConfig(temperature=0.0),
        )
        return result.to_evaluations()
    except Exception as e:
        return DeepSearchQAResult.error_evaluations(str(e))


console.print('[green]✓[/green] agent_task and deepsearchqa_evaluator defined')

## 3. Define Trace-Level Evaluators

Each trace evaluator receives the full Langfuse trace *after* the experiment completes.
It inspects the raw tool call sequence — no LLM needed, just deterministic rules.

These run in a **second pass** after all agent answers are collected, so the agent isn't slowed down.

In [ ]:
def trace_tool_call_count(*, trace, item_result, **kwargs):
    """
    Count total tool calls made during the agent run.
    More calls generally means more thorough research — but too many may indicate redundancy.
    """
    metrics = extract_trace_metrics(trace)
    return [Evaluation(name='trace_tool_call_count', value=float(metrics.tool_call_count))]


def trace_search_before_fetch(*, trace, item_result, **kwargs):
    """
    Check that google_search always precedes web_fetch.
    Score 1.0 = agent always searches before fetching (correct research pattern).
    Score 0.0 = agent fetched a URL without first searching (bad pattern).
    """
    observations = trace.observations or []
    tool_names = [
        (obs.name or '').lower()
        for obs in observations
        if 'tool' in (obs.type or '').lower() or 'tool' in (obs.name or '').lower()
    ]

    seen_search = False
    violation = False
    for name in tool_names:
        if 'google_search' in name or 'search' in name:
            seen_search = True
        elif 'web_fetch' in name or 'fetch' in name:
            if not seen_search:
                violation = True
                break

    return [Evaluation(
        name='trace_search_before_fetch',
        value=0.0 if violation else 1.0,
        comment='1.0 = agent always searched before fetching'
    )]


def trace_has_sources(*, trace, item_result, **kwargs):
    """
    Check whether the agent cited at least one source URL.
    Score 1.0 = sources found (trustworthy answer).
    Score 0.0 = no sources (answer may be unreliable).
    """
    output_text = str(trace.output or '')
    has_url = 'http' in output_text or 'www.' in output_text

    metadata = trace.metadata or {}
    sources = metadata.get('sources', [])
    has_sources_meta = isinstance(sources, list) and len(sources) > 0

    score = 1.0 if (has_url or has_sources_meta) else 0.0
    return [Evaluation(name='trace_has_sources', value=score)]


def trace_redundant_search_ratio(*, trace, item_result, **kwargs):
    """
    Measure the fraction of duplicate search queries.
    0.0 = no redundancy (ideal — every search added new information).
    1.0 = all searches were repeats (completely wasteful).
    """
    observations = trace.observations or []
    search_queries = []

    for obs in observations:
        name = (obs.name or '').lower()
        if 'search' in name:
            raw_input = obs.input
            if isinstance(raw_input, dict):
                query = raw_input.get('query') or raw_input.get('q') or ''
            else:
                query = str(raw_input or '')
            if query.strip():
                search_queries.append(query.strip().lower())

    if not search_queries:
        return [Evaluation(name='trace_redundant_search_ratio', value=0.0)]

    unique = len(set(search_queries))
    redundant = len(search_queries) - unique
    ratio = redundant / len(search_queries)

    return [Evaluation(
        name='trace_redundant_search_ratio',
        value=ratio,
        metadata={'total_searches': len(search_queries), 'unique_searches': unique}
    )]


def trace_latency_sec(*, trace, item_result, **kwargs):
    """Record total agent wall-clock time in seconds."""
    metrics = extract_trace_metrics(trace)
    if metrics.latency_sec is None:
        return []
    return [Evaluation(name='trace_latency_sec', value=metrics.latency_sec)]


TRACE_EVALUATORS = [
    trace_tool_call_count,
    trace_search_before_fetch,
    trace_has_sources,
    trace_redundant_search_ratio,
    trace_latency_sec,
]

console.print(f'[green]✓[/green] {len(TRACE_EVALUATORS)} trace evaluators defined')

## 4. Run the Two-Pass Experiment

`run_experiment_with_trace_evals` does two things:

- **Pass 1:** Runs the agent on every dataset item → scores with `deepsearchqa_evaluator`
- **Pass 2:** Waits for Langfuse traces to be fully indexed → runs all 5 trace evaluators

The trace wait (`max_wait_sec=180`) is needed because Langfuse indexes traces asynchronously.

> **Time estimate:** 5 questions × ~60s each = ~5 min for Pass 1, plus ~1 min for Pass 2.

In [ ]:
result = await run_experiment_with_trace_evals(
    DATASET_NAME,
    name=EXPERIMENT_NAME,
    task=agent_task,
    evaluators=[deepsearchqa_evaluator],
    trace_evaluators=TRACE_EVALUATORS,
    trace_wait=TraceWaitConfig(max_wait_sec=180),
    description=f'TR Trajectory Eval — {CATEGORY} — {NUM_SAMPLES} samples',
    max_concurrency=1,
)

console.print('[green]✓[/green] Two-pass experiment complete!')
if result.experiment.dataset_run_url:
    display(HTML(f'<p>View in Langfuse: <a href="{result.experiment.dataset_run_url}" target="_blank">{result.experiment.dataset_run_url}</a></p>'))

## 5. Inspect Results

### 5.1 Output-Level Scores (Answer Quality)

In [ ]:
rows = []
for item_result in result.experiment.item_results:
    question = str(item_result.item.input)
    row = {'question': question[:55] + '...' if len(question) > 55 else question}
    for evaluation in item_result.evaluations or []:
        row[evaluation.name] = evaluation.value
    rows.append(row)

df_output = pd.DataFrame(rows)
print(df_output.to_string(index=False))

numeric_cols = [c for c in ['F1', 'Precision', 'Recall'] if c in df_output.columns]
if numeric_cols:
    t = Table(title='Mean Output Scores (Answer Quality)')
    t.add_column('Metric', style='cyan')
    t.add_column('Mean', style='white')
    for col in numeric_cols:
        t.add_row(col, f'{df_output[col].mean():.3f}')
    console.print(t)

### 5.2 Trace-Level Scores (Trajectory / Behavior Quality)

In [ ]:
trace_evals = result.trace_evaluations
trace_scores = trace_evals.evaluations_by_trace_id

trace_rows = []
for item_result in result.experiment.item_results:
    question = str(item_result.item.input)
    row = {'question': question[:50] + '...' if len(question) > 50 else question}
    trace_id = item_result.trace_id
    if trace_id and trace_id in trace_scores:
        for ev in trace_scores[trace_id]:
            row[ev.name] = ev.value
    trace_rows.append(row)

df_trace = pd.DataFrame(trace_rows)
print(df_trace.to_string(index=False))

trace_metric_cols = [
    c for c in [
        'trace_tool_call_count', 'trace_search_before_fetch',
        'trace_has_sources', 'trace_redundant_search_ratio', 'trace_latency_sec'
    ] if c in df_trace.columns
]

if trace_metric_cols:
    interpretations = {
        'trace_tool_call_count':        'avg tool calls per question',
        'trace_search_before_fetch':    '1.0 = always searched first',
        'trace_has_sources':            '1.0 = always cited sources',
        'trace_redundant_search_ratio': '0.0 = no repeated searches',
        'trace_latency_sec':            'avg seconds per question',
    }
    t = Table(title='Mean Trace Scores (Trajectory Quality)')
    t.add_column('Metric', style='cyan')
    t.add_column('Mean', style='white')
    t.add_column('Ideal', style='dim')
    t.add_column('Interpretation', style='dim')
    ideals = {
        'trace_tool_call_count': '4–8',
        'trace_search_before_fetch': '1.0',
        'trace_has_sources': '1.0',
        'trace_redundant_search_ratio': '0.0',
        'trace_latency_sec': '<90s',
    }
    for col in trace_metric_cols:
        t.add_row(col, f'{df_trace[col].mean():.3f}', ideals.get(col, '?'), interpretations.get(col, ''))
    console.print(t)

console.print(f'[dim]Skipped traces: {len(trace_evals.skipped_trace_ids)}  Failed: {len(trace_evals.failed_trace_ids)}[/dim]')

### 5.3 Combined View — Answer Quality vs Behavior Quality

In [ ]:
# Merge output and trace scores on question
df_combined = df_output.merge(df_trace, on='question', how='outer')
print(df_combined.to_string(index=False))

# Highlight any questions where good F1 but bad trace (or vice versa)
if 'F1' in df_combined.columns and 'trace_has_sources' in df_combined.columns:
    good_answer_no_sources = df_combined[
        (df_combined['F1'] > 0.5) & (df_combined['trace_has_sources'] == 0.0)
    ]
    if not good_answer_no_sources.empty:
        console.print('[yellow]⚠ Questions with good F1 but no sources cited:[/yellow]')
        print(good_answer_no_sources[['question', 'F1', 'trace_has_sources']].to_string(index=False))

## 6. Iterate & Experiment

Update `EXPERIMENT_NAME` in Cell 1 and re-run from there to create a new named experiment.
Compare experiments side-by-side in Langfuse under **Datasets → TR-DeepSearchQA**.

### Levers to pull:

| Change | How | Watch for |
|--------|-----|----------|
| Disable planning | `KnowledgeGroundedAgent(enable_planning=False)` | Drop in F1? Fewer tool calls? |
| Change category | `CATEGORY = 'Science & Technology'` | Which domains have worse trajectory? |
| Add a 6th trace evaluator | Write a new `def trace_xxx(...)` function | E.g. count file tool usage |
| Adjust redundancy threshold | Modify `trace_redundant_search_ratio` logic | What counts as redundant? |
| Stronger model | Set `DEFAULT_PLANNER_MODEL=gemini-2.5-pro` in `.env` | Better F1? Longer latency? |

### Adding your own trace evaluator:

```python
def trace_file_tool_usage(*, trace, item_result, **kwargs):
    """Count how many times the agent used file tools (fetch_file, grep_file, read_file)."""
    observations = trace.observations or []
    file_calls = sum(
        1 for obs in observations
        if any(t in (obs.name or '').lower() for t in ['fetch_file', 'grep_file', 'read_file'])
    )
    return [Evaluation(name='trace_file_tool_usage', value=float(file_calls))]

# Add to TRACE_EVALUATORS list and re-run the experiment
```

In [ ]:
console.print(Panel(
    '[green]✓[/green] Notebook 03 complete!\n\n'
    'You now have both output-level and trace-level scores in Langfuse.\n'
    'Use the Datasets tab to compare experiments across runs.',
    title='Done',
    border_style='green',
))